In [4]:
""" import joblib

params = joblib.load(
    "../../data/processed/CTB/prosit_params/s5_baseline_with_binning_params.pkl"
)

print(type(params))

print(
    [m for m in dir(params)
     if not m.startswith("_")]
) """

' import joblib\n\nparams = joblib.load(\n    "../../data/processed/CTB/prosit_params/s5_baseline_with_binning_params.pkl"\n)\n\nprint(type(params))\n\nprint(\n    [m for m in dir(params)\n     if not m.startswith("_")]\n) '

In [5]:
""" import joblib
import pandas as pd

from prosit.simulator import Simulator

# ==========================================================
# LOAD PARAMETERS
# ==========================================================

params = joblib.load(
    "../../data/processed/CTB/prosit_params/s5_baseline_with_binning_params.pkl"
)

print(type(params))

# ==========================================================
# INITIALISE SIMULATOR
# ==========================================================

simulator = Simulator(
    params
)

# ==========================================================
# GENERATE 20.000 CASES
# ==========================================================

sim_log = simulator.simulate(
    n_cases=20_000
)

# ==========================================================
# DATAFRAME
# ==========================================================

sim_df = pd.DataFrame(sim_log)

print(
    f"Generated events: {len(sim_df):,}"
)

print(
    f"Generated cases: "
    f"{sim_df['case:concept:name'].nunique():,}"
)

sim_df.head() """

' import joblib\nimport pandas as pd\n\nfrom prosit.simulator import Simulator\n\n# ==========================================================\n# LOAD PARAMETERS\n# ==========================================================\n\nparams = joblib.load(\n    "../../data/processed/CTB/prosit_params/s5_baseline_with_binning_params.pkl"\n)\n\nprint(type(params))\n\n# ==========================================================\n# INITIALISE SIMULATOR\n# ==========================================================\n\nsimulator = Simulator(\n    params\n)\n\n# ==========================================================\n# GENERATE 20.000 CASES\n# ==========================================================\n\nsim_log = simulator.simulate(\n    n_cases=20_000\n)\n\n# ==========================================================\n# DATAFRAME\n# ==========================================================\n\nsim_df = pd.DataFrame(sim_log)\n\nprint(\n    f"Generated events: {len(sim_df):,}"\n)\n\nprint(\n    f"

In [6]:
#model.write_dot("rmg_receive.dot")

In [7]:
import pandas as pd
import numpy as np

from scipy.stats import wasserstein_distance

In [8]:
REFERENCE_CSV = (
    "../../data/processed/CTB/sampled_real_eventlogs/s5_sample_20.000_eventlog_one_block_binned_target_features.csv"
)

SIM_CSV = (
    "../../data/processed/CTB/prosit_simulations/sim_log_s5_baseline_sample_20.000_v3_binning.csv"
)

ref = pd.read_csv(REFERENCE_CSV)
sim = pd.read_csv(SIM_CSV)

print(
    f"Reference events: {len(ref):,}"
)

print(
    f"Simulation events: {len(sim):,}"
)

Reference events: 61,718
Simulation events: 48,290


In [9]:
# ==========================================================
# CALCULATE SIMULATION KPIS
# ==========================================================

for col in [
    "enabled:timestamp",
    "start:timestamp",
    "time:timestamp"
]:
    sim[col] = pd.to_datetime(sim[col])

# ----------------------------------------------------------
# EVENT LEVEL KPIs
# ----------------------------------------------------------

sim["waiting_time"] = (
    sim["start:timestamp"]
    - sim["enabled:timestamp"]
).dt.total_seconds() / 60

sim["service_time"] = (
    sim["time:timestamp"]
    - sim["start:timestamp"]
).dt.total_seconds() / 60

# ----------------------------------------------------------
# CASE LEVEL KPI
# ----------------------------------------------------------

case_start = (
    sim.groupby(
        "case:concept:name"
    )["start:timestamp"]
    .transform("min")
)

case_end = (
    sim.groupby(
        "case:concept:name"
    )["time:timestamp"]
    .transform("max")
)

sim["turnaround_time"] = (
    case_end
    - case_start
).dt.total_seconds() / 60

# ----------------------------------------------------------
# RMG EVENTS ONLY
# ----------------------------------------------------------

rmg_activities = [
    "RMG_receive",
    "RMG_delivery",
    "RMG_mixed"
]

sim_rmg = sim[
    sim["concept:name"]
    .isin(rmg_activities)
].copy()

ref_rmg = ref[
    ref["concept:name"]
    .isin(rmg_activities)
].copy()

print(
    f"Reference RMG events: {len(ref_rmg):,}"
)

print(
    f"Simulation RMG events: {len(sim_rmg):,}"
)

Reference RMG events: 20,000
Simulation RMG events: 16,071


In [10]:
#MAP RECEIVE; DELIVER & MIXED
def add_operation_type(df):

    df = df.copy()

    df["operation_type"] = np.where(
        df["concept:name"].str.contains(
            "receive",
            case=False,
            na=False
        ),
        "receive",
        np.where(
            df["concept:name"].str.contains(
                "delivery",
                case=False,
                na=False
            ),
            "delivery",
            np.where(
                df["concept:name"].str.contains(
                    "mixed",
                    case=False,
                    na=False
                ),
                "mixed",
                np.nan
            )
        )
    )

    return df

ref = add_operation_type(ref)
sim = add_operation_type(sim)

In [11]:
#COMPARING FUNCTION'
def evaluate_kpi(
    ref_values,
    sim_values,
    kpi_name,
    segment
):

    ref_values = (
        pd.Series(ref_values)
        .dropna()
    )

    sim_values = (
        pd.Series(sim_values)
        .dropna()
    )

    return {
        "segment": segment,
        "kpi": kpi_name,

        "ref_mean":
        ref_values.mean(),

        "sim_mean":
        sim_values.mean(),

        "ref_median":
        ref_values.median(),

        "sim_median":
        sim_values.median(),

        "ref_std":
        ref_values.std(),

        "sim_std":
        sim_values.std(),

        "ref_p95":
        ref_values.quantile(0.95),

        "sim_p95":
        sim_values.quantile(0.95),

        "wasserstein":
        wasserstein_distance(
            ref_values,
            sim_values
        )
    }

In [12]:
# ==========================================================
# CASE LEVEL TURNAROUND VALIDATION
# ==========================================================

ref_cases = (
    ref.groupby(
        "case:concept:name"
    )["turnaround_time"]
    .max()
)

sim_cases = (
    sim.groupby(
        "case:concept:name"
    )["turnaround_time"]
    .max()
)

case_eval = pd.DataFrame(
    [
        {
            "kpi": "turnaround_time",

            "ref_mean":
            ref_cases.mean(),

            "sim_mean":
            sim_cases.mean(),

            "ref_median":
            ref_cases.median(),

            "sim_median":
            sim_cases.median(),

            "wasserstein":
            wasserstein_distance(
                ref_cases,
                sim_cases
            )
        }
    ]
)

case_eval

,kpi,ref_mean,sim_mean,ref_median,sim_median,wasserstein
0,turnaround_time,37.37715,55.379609,30.0,31.0,18.260615


In [13]:
# WHOLE EVALUATION
results = []
segments = {

    "all_rmg": (
        ref_rmg,
        sim_rmg
    ),

    "receive": (
        ref_rmg[
            ref_rmg["concept:name"]
            == "RMG_receive"
        ],
        sim_rmg[
            sim_rmg["concept:name"]
            == "RMG_receive"
        ]
    ),

    "delivery": (
        ref_rmg[
            ref_rmg["concept:name"]
            == "RMG_delivery"
        ],
        sim_rmg[
            sim_rmg["concept:name"]
            == "RMG_delivery"
        ]
    ),

    "mixed": (
        ref_rmg[
            ref_rmg["concept:name"]
            == "RMG_mixed"
        ],
        sim_rmg[
            sim_rmg["concept:name"]
            == "RMG_mixed"
        ]
    )
}

for segment, (
    ref_seg,
    sim_seg
) in segments.items():

    for kpi in [
        "waiting_time",
        "service_time",
        "turnaround_time"
    ]:

        results.append(

            evaluate_kpi(
                ref_seg[kpi],
                sim_seg[kpi],
                kpi,
                segment
            )

        )

evaluation = pd.DataFrame(
    results
)

In [14]:
# FINAL EVALUATION TABLE
evaluation = evaluation.round(2)

evaluation = evaluation.sort_values(
    [
        "segment",
        "kpi"
    ]
)

evaluation

,segment,kpi,ref_mean,sim_mean,ref_median,sim_median,ref_std,sim_std,ref_p95,sim_p95,wasserstein
1,all_rmg,service_time,11.71,16.25,8.0,9.0,11.43,98.09,32.00,33.00,4.63
2,all_rmg,turnaround_time,37.38,55.72,30.0,32.0,27.77,205.43,85.00,97.76,18.56
0,all_rmg,waiting_time,7.28,25.70,3.0,7.0,13.98,174.37,32.00,52.06,18.42
7,delivery,service_time,8.47,12.60,6.0,5.0,8.55,96.12,23.00,26.00,5.39
8,delivery,turnaround_time,29.69,57.97,23.0,29.0,25.27,236.43,67.45,92.26,28.62
6,delivery,waiting_time,6.08,32.39,3.0,7.0,9.03,216.40,24.00,56.15,26.31
10,mixed,service_time,14.86,23.52,11.0,11.0,13.38,140.48,39.00,46.00,9.12
11,mixed,turnaround_time,41.03,63.67,34.0,35.0,27.73,228.00,91.70,109.72,23.17
9,mixed,waiting_time,7.97,26.59,3.0,7.0,14.88,179.56,33.00,55.00,18.62
4,receive,service_time,12.15,15.04,9.0,9.0,11.53,85.85,32.00,31.00,3.53


In [15]:
# QUICK OVERVIEW ONLY WASSERSETEIN
evaluation.pivot(
    index="segment",
    columns="kpi",
    values="wasserstein"
).round(2)

kpi,service_time,turnaround_time,waiting_time
segment,,,
all_rmg,4.63,18.56,18.42
delivery,5.39,28.62,26.31
mixed,9.12,23.17,18.62
receive,3.53,16.12,17.10


COMPARING STRUCTURAL PATTERNS

In [17]:
# ==========================================================
# ACTIVITY COUNTS
# ==========================================================

ref_acts = (
    ref["concept:name"]
    .value_counts()
    .rename("reference")
)

sim_acts = (
    sim["concept:name"]
    .value_counts()
    .rename("simulation")
)

activity_comparison = pd.concat(
    [ref_acts, sim_acts],
    axis=1
).fillna(0)

activity_comparison["diff"] = (
    activity_comparison["simulation"]
    - activity_comparison["reference"]
)

activity_comparison["diff_pct"] = (
    100
    * activity_comparison["diff"]
    / activity_comparison["reference"]
)

activity_comparison.sort_index()

,reference,simulation,diff,diff_pct
Gate In,20000,16000,-4000,-20.000000
Gate Out,20000,16000,-4000,-20.000000
HO2_delivery,136,108,-28,-20.588235
HO2_mixed,34,33,-1,-2.941176
HO2_receive,43,41,-2,-4.651163
LL_delivery,994,2,-992,-99.798793
LL_mixed,473,1,-472,-99.788584
LL_receive,38,34,-4,-10.526316
RMG_delivery,5112,1412,-3700,-72.378717
RMG_mixed,3667,2702,-965,-26.315789


In [18]:
# ==========================================================
# RESOURCE DISTRIBUTIONS COUNTS
# ==========================================================

ref_res = (
    ref["org:resource"]
    .value_counts()
    .rename("reference")
)

sim_res = (
    sim["org:resource"]
    .value_counts()
    .rename("simulation")
)

resource_comparison = pd.concat(
    [ref_res, sim_res],
    axis=1
).fillna(0)

resource_comparison["diff"] = (
    resource_comparison["simulation"]
    - resource_comparison["reference"]
)

resource_comparison["diff_pct"] = (
    100
    * resource_comparison["diff"]
    / resource_comparison["reference"]
)

resource_comparison.sort_index()

,reference,simulation,diff,diff_pct
HO2,213,182,-31,-14.553991
LL,1505,37,-1468,-97.541528
Res.GateIn,20000,16000,-4000,-20.000000
Res.GateOut,20000,16000,-4000,-20.000000
T06,874,593,-281,-32.151030
T07,750,484,-266,-35.466667
T08,734,488,-246,-33.514986
T09,791,589,-202,-25.537295
T10,876,594,-282,-32.191781
T11,923,651,-272,-29.469122


In [19]:
# ==========================================================
# PROCESS TYPE COUNTS
# ==========================================================

rmg_types = [
    "RMG_receive",
    "RMG_delivery",
    "RMG_mixed"
]

ref_proc = (
    ref[
        ref["concept:name"].isin(rmg_types)
    ]["concept:name"]
    .value_counts()
    .rename("reference")
)

sim_proc = (
    sim[
        sim["concept:name"].isin(rmg_types)
    ]["concept:name"]
    .value_counts()
    .rename("simulation")
)

process_comparison = pd.concat(
    [ref_proc, sim_proc],
    axis=1
)

process_comparison["diff_pct"] = (
    100
    * (
        process_comparison["simulation"]
        - process_comparison["reference"]
    )
    / process_comparison["reference"]
)

process_comparison

,reference,simulation,diff_pct
RMG_receive,11221,11957,6.559130
RMG_delivery,5112,1412,-72.378717
RMG_mixed,3667,2702,-26.315789


In [20]:
#===========================
# MEAN ERROR
#===========================
evaluation["mean_error_pct"] = (
    100
    * (
        evaluation["sim_mean"]
        - evaluation["ref_mean"]
    )
    / evaluation["ref_mean"]
)

evaluation[
    [
        "segment",
        "kpi",
        "mean_error_pct"
    ]
]

,segment,kpi,mean_error_pct
1,all_rmg,service_time,38.770282
2,all_rmg,turnaround_time,49.063670
0,all_rmg,waiting_time,253.021978
7,delivery,service_time,48.760331
8,delivery,turnaround_time,95.250926
6,delivery,waiting_time,432.730263
10,mixed,service_time,58.277254
11,mixed,turnaround_time,55.179137
9,mixed,waiting_time,233.626098
4,receive,service_time,23.786008


In [21]:
#================
# MEDIAN ERROR 
#==========================

evaluation["median_error_pct"] = (
    100
    * (
        evaluation["sim_median"]
        - evaluation["ref_median"]
    )
    / evaluation["ref_median"]
)

evaluation[
    [
        "segment",
        "kpi",
        "median_error_pct"
    ]
]


,segment,kpi,median_error_pct
1,all_rmg,service_time,12.500000
2,all_rmg,turnaround_time,6.666667
0,all_rmg,waiting_time,133.333333
7,delivery,service_time,-16.666667
8,delivery,turnaround_time,26.086957
6,delivery,waiting_time,133.333333
10,mixed,service_time,0.000000
11,mixed,turnaround_time,2.941176
9,mixed,waiting_time,133.333333
4,receive,service_time,0.000000


In [22]:
#====================
# FINAL SUMMARY TABLE
#=========================
validation_summary = evaluation[
    [
        "segment",
        "kpi",
        "wasserstein"
    ]
].copy()

validation_summary["mean_error_pct"] = (
    100
    * (
        evaluation["sim_mean"]
        - evaluation["ref_mean"]
    )
    / evaluation["ref_mean"]
)

validation_summary["median_error_pct"] = (
    100
    * (
        evaluation["sim_median"]
        - evaluation["ref_median"]
    )
    / evaluation["ref_median"]
)

validation_summary.round(2)

,segment,kpi,wasserstein,mean_error_pct,median_error_pct
1,all_rmg,service_time,4.63,38.77,12.50
2,all_rmg,turnaround_time,18.56,49.06,6.67
0,all_rmg,waiting_time,18.42,253.02,133.33
7,delivery,service_time,5.39,48.76,-16.67
8,delivery,turnaround_time,28.62,95.25,26.09
6,delivery,waiting_time,26.31,432.73,133.33
10,mixed,service_time,9.12,58.28,0.00
11,mixed,turnaround_time,23.17,55.18,2.94
9,mixed,waiting_time,18.62,233.63,133.33
4,receive,service_time,3.53,23.79,0.00
